In [1]:
import torch
import torch.nn as nn
from fla.layers import MultiScaleRetention

torch.manual_seed(0)

/home/shida/miniconda3/envs/fla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
batch_size, num_heads, seq_len, hidden_size,  = 1, 4, 2048, 1024
device, dtype = 'cuda:0', torch.bfloat16
retnet = MultiScaleRetention(mode='chunk', hidden_size=hidden_size, num_heads=num_heads).to(device=device, dtype=dtype)
x = torch.randn(batch_size, seq_len, hidden_size).to(device=device, dtype=dtype)
y, *_ = retnet(x)
y.shape

Using chunk


torch.Size([1, 2048, 1024])

In [3]:
# Randomly generate x and y tensors
x = torch.randn(batch_size, seq_len, hidden_size).to(device=device, dtype=dtype)
y_true = torch.randn(batch_size, seq_len, hidden_size).to(device=device, dtype=dtype)

# Forward pass to get the model output
y_pred, *_ = retnet(x)

# Define the mean square error loss
criterion = nn.MSELoss()

# Compute the loss
loss = criterion(y_pred, y_true)

# Backward pass to compute gradients
loss.backward()

# Compute the norm of the gradients
total_norm = 0.0
for name, param in retnet.named_parameters():
    if param.grad is not None:
        param_norm = param.grad.data.norm(2)
        total_norm += param_norm.item() ** 2

total_norm = total_norm ** 0.5

# Print the shape of the model output, the loss value, and the gradient norm
print(f'y_pred shape: {y_pred.shape}')
print(f'Loss: {loss.item()}')
print(f'Total gradient norm: {total_norm}')

Using chunk
y_pred shape: torch.Size([1, 2048, 1024])
Loss: 1.0
Total gradient norm: 0.009479828425803366
